In [0]:
%sql
USE CATALOG V_Commerce;

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
catalog = "v_commerce"
gold_schema_name = "gold"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
tb_avaliacoes_silver = spark.table("v_commerce.silver.tb_avaliacoes")
tb_catalogo_produtos_silver = spark.table("v_commerce.silver.tb_catalogo_produtos")
tb_clickstream_silver = spark.table("v_commerce.silver.tb_clickstream")
tb_clientes_silver = spark.table("v_commerce.silver.tb_clientes")
tb_pedidos_silver = spark.table("v_commerce.silver.tb_pedidos")
tb_suporte_tickets_silver = spark.table("v_commerce.silver.tb_suporte_tickets") 

In [0]:
# gold.fato_avaliacoes_pedido
# Origem: silver.tb_avaliacoes + silver.tb_pedidos + silver.tb_catalogo_produtos

avaliacoes  = spark.table("v_commerce.silver.tb_avaliacoes")
pedidos     = spark.table("v_commerce.silver.tb_pedidos")
catalogo    = spark.table("v_commerce.silver.tb_catalogo_produtos") \
                   .select("id_produto", "nome_produto", "categoria", "preco")

# ── pct_recomendacoes_sim por produto ─────────────────────────────────────────
pct_recomendacoes = (
    avaliacoes
    .groupBy("id_produto")
    .agg(
        F.round(
            F.sum(F.when(F.col("recomenda") == "Sim", 1).otherwise(0)) /
            F.count("id_avaliacao") * 100, 2
        ).alias("pct_recomendacoes_sim")
    )
)

fato_avaliacoes_pedido = (
    avaliacoes
    .join(pedidos.select(
        "id_pedido", "valor_pedido", "data_pedido",
        "metodo_pagamento", "status", "quantidade"
    ), on="id_pedido", how="left")

    .join(catalogo, on="id_produto", how="left")
    .join(pct_recomendacoes, on="id_produto", how="left")

    # ── gera sequencial ordenado por data do pedido ───────────────────────────
    .withColumn(
        "_seq",
        F.lpad(
            F.row_number().over(Window.orderBy("data_pedido")).cast("string"),
            8, "0"
        )
    )

    .select(
        # chaves
        F.col("id_avaliacao"),
        F.col("id_pedido"),

        # id amigável para frontend: PED - 0000 0000
        F.concat(
            F.lit("PED - "),
            F.substring(F.col("_seq"), 1, 4),
            F.lit(" "),
            F.substring(F.col("_seq"), 5, 4),
        ).alias("id_pedido_display"),

        F.col("id_cliente"),
        F.col("id_produto"),

        # dimensões do produto
        F.col("nome_produto"),
        F.col("categoria"),
        F.col("preco"),

        # dimensões do pedido
        F.col("valor_pedido"),
        F.col("quantidade"),
        F.col("metodo_pagamento"),
        F.col("status"),
        F.col("data_pedido"),

        # métricas de avaliação
        F.col("nota_produto"),
        F.col("nota_nps"),
        F.col("recomenda"),
        F.col("comentario"),
        F.col("data_avaliacao"),

        # categoria NPS derivada da nota_nps
        F.when(F.col("nota_nps").between(9, 10), "Promotor")
         .when(F.col("nota_nps").between(7,  8), "Neutro")
         .when(F.col("nota_nps").between(0,  6), "Detrator")
         .otherwise(None)
         .alias("categoria_nps"),

        # percentual de recomendações "Sim" para o produto
        F.col("pct_recomendacoes_sim"),

        F.current_timestamp().alias("timestamp_ingestion_gold"),
    )
)

fato_avaliacoes_pedido.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.fato_avaliacoes_pedido")

print(f"✅ Tabela {catalog}.{gold_schema_name}.fato_avaliacoes_pedido criada com sucesso!")
print(f"   Total de linhas: {fato_avaliacoes_pedido.count()}")

✅ Tabela v_commerce.gold.fato_avaliacoes_pedido criada com sucesso!
   Total de linhas: 156832


In [0]:
# gold.dim_produto
# Origem: silver.tb_catalogo_produtos + silver.tb_pedidos
#         + silver.tb_suporte_tickets + silver.tb_avaliacoes

catalogo = spark.table("v_commerce.silver.tb_catalogo_produtos")
pedidos = spark.table("v_commerce.silver.tb_pedidos")
tickets = spark.table("v_commerce.silver.tb_suporte_tickets")
avaliacoes = spark.table("v_commerce.silver.tb_avaliacoes")

# ── Métricas de vendas por produto ────────────────────────────────────────────
metricas_pedidos = (
    pedidos
    .groupBy("id_produto")
    .agg(
        F.count("id_pedido")                    .alias("total_pedidos"),
        F.round(F.sum("valor_pedido"), 2)        .alias("receita_total"),
        F.round(F.avg("valor_pedido"), 2)        .alias("ticket_medio"),
        F.sum("quantidade")                      .alias(
            "total_unidades_vendidas"),
    )
)

# ── Métricas de avaliação por produto ─────────────────────────────────────────
metricas_avaliacoes = (
    avaliacoes
    .groupBy("id_produto")
    .agg(
        F.count("id_avaliacao")                                                      .alias(
            "total_avaliacoes"),
        F.round(F.avg("nota_produto"), 2)                                            .alias(
            "media_nota_produto"),
        F.round(F.avg("nota_nps"), 2)                                                .alias(
            "media_nota_nps"),
        F.round(
            F.sum(F.when(F.col("recomenda") == "Sim", 1).otherwise(0)) /
            F.count("id_avaliacao") * 100, 2
        )                                                                            .alias("pct_recomendacoes_sim"),
    )
)

# ── Métricas de suporte por produto ───────────────────────────────────────────
# tickets não tem id_produto direto — liga via id_pedido
metricas_tickets = (
    tickets
    .join(pedidos.select("id_pedido", "id_produto"), on="id_pedido", how="left")
    .groupBy("id_produto")
    .agg(
        F.count("id_ticket")                          .alias("total_tickets"),
        F.round(F.avg("tempo_resolucao_horas"), 2)    .alias(
            "media_tempo_resolucao_horas"),
        F.round(F.avg("nota_avaliacao"), 2)            .alias(
            "media_nota_suporte"),
    )
)

# ── Montagem final da dim_produto ─────────────────────────────────────────────
dim_produto = (
    catalogo
    .join(metricas_pedidos,    on="id_produto", how="left")
    .join(metricas_avaliacoes, on="id_produto", how="left")
    .join(metricas_tickets,    on="id_produto", how="left")

    .select(
        # identificação
        F.col("id_produto"),
        F.col("sku"),
        F.col("nome_produto"),
        F.col("categoria"),
        F.col("fornecedor"),

        # preço e estoque
        F.col("preco"),
        F.col("peso_kg"),
        F.col("estoque_disponivel"),
        F.col("ativo"),

        # flags de qualidade
        F.col("precisa_revisao"),

        # datas
        F.col("data_cadastro_produto"),

        # métricas de vendas
        F.coalesce(F.col("total_pedidos"),
                   F.lit(0)).alias("total_pedidos"),
        F.coalesce(F.col("receita_total"),
                   F.lit(0)).alias("receita_total"),
        F.coalesce(F.col("ticket_medio"),
                   F.lit(0)).alias("ticket_medio"),
        F.coalesce(F.col("total_unidades_vendidas"), F.lit(0)
                   ).alias("total_unidades_vendidas"),

        # métricas de avaliação
        F.coalesce(F.col("total_avaliacoes"),        F.lit(0)
                   )   .alias("total_avaliacoes"),
        F.col("media_nota_produto"),
        F.col("media_nota_nps"),
        F.col("pct_recomendacoes_sim"),

        # métricas de suporte
        F.coalesce(F.col("total_tickets"),
                   F.lit(0))   .alias("total_tickets"),
        F.col("media_tempo_resolucao_horas"),
        F.col("media_nota_suporte"),

        F.current_timestamp().alias("timestamp_ingestion_gold"),
    )
)

dim_produto.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.dim_produto")

print(f"✅ Tabela {catalog}.{gold_schema_name}.dim_produto criada com sucesso!")
print(f"   Total de linhas: {dim_produto.count()}")

✅ Tabela v_commerce.gold.dim_produto criada com sucesso!
   Total de linhas: 517


In [0]:
# gold.dim_tempo
# Gerada sinteticamente a partir do intervalo de datas do dataset
# Fonte de referência: tb_pedidos_silver (data_pedido)

pedidos = spark.table("v_commerce.silver.tb_pedidos")

# ── Extrai intervalo de datas do dataset ──────────────────────────────────────
datas = pedidos.agg(
    F.min("data_pedido").alias("data_min"),
    F.max("data_pedido").alias("data_max")
).collect()[0]

data_min = datas["data_min"]
data_max = datas["data_max"]

print(f"📅 Intervalo de datas: {data_min} → {data_max}")

# ── Gera sequência de datas entre min e max ───────────────────────────────────
dim_tempo = (
    spark.sql(f"""
        SELECT explode(sequence(
            date '{data_min}',
            date '{data_max}',
            interval 1 day
        )) AS id_data
    """)

    .select(
        # chave
        F.col("id_data"),

        # extrações diretas
        F.year("id_data")                          .alias("ano"),
        F.month("id_data")                         .alias("mes"),
        F.dayofmonth("id_data")                    .alias("dia"),
        F.quarter("id_data")                       .alias("trimestre"),
        F.dayofweek("id_data")                     .alias("dia_semana_num"),
        F.dayofyear("id_data")                     .alias("dia_do_ano"),
        F.weekofyear("id_data")                    .alias("semana_do_ano"),

        # descrições
        F.date_format("id_data", "MMMM")           .alias("nome_mes"),
        F.date_format("id_data", "EEEE")           .alias("nome_dia_semana"),
        F.date_format("id_data", "yyyy-MM")        .alias("ano_mes"),
        F.concat(F.lit("Q"), F.quarter("id_data")) .alias("trimestre_label"),

        # flags úteis
        F.when(F.dayofweek("id_data").isin(1, 7), "Sim")
         .otherwise("Nao")
         .alias("fim_de_semana"),

        F.current_timestamp()                   .alias("timestamp_ingestion_gold"),
    )
)

dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.dim_tempo")

print(f"✅ Tabela {catalog}.{gold_schema_name}.dim_tempo criada com sucesso!")
print(f"   Total de dias gerados: {dim_tempo.count()}")

📅 Intervalo de datas: 2023-01-01 → 2026-05-31
✅ Tabela v_commerce.gold.dim_tempo criada com sucesso!
   Total de dias gerados: 1247


In [0]:
# gold.fato_clickstream_navegacao

df_agregado = tb_clickstream_silver.groupBy("id_cliente").agg(
    F.countDistinct("id_sessao").alias("total_sessoes"),
    F.count("id_evento").alias("total_eventos"),
    F.max("data_evento").alias("data_ultima_sessao"),

    F.count(F.when(F.col("tipo_evento") == "product_view", 1)
            ).alias("qtd_visualizacao_produto"),
    F.count(F.when(F.col("tipo_evento") == "add_to_cart", 1)
            ).alias("qtd_adicoes_carrinho"),
    F.count(F.when(F.col("tipo_evento") == "abandon_cart", 1)
            ).alias("qtd_abandonos_carrinho"),
    F.count(F.when(F.col("tipo_evento") == "purchase", 1)).alias("qtd_compras")
)


def get_most_frequent(column_name):
    window = Window.partitionBy("id_cliente").orderBy(
        F.desc("count"), F.desc(column_name))
    return tb_clickstream_silver.groupBy("id_cliente", column_name).count().withColumn("rn", F.row_number().over(window)).filter("rn = 1").select("id_cliente", F.col(column_name).alias(f"{column_name}_mais_usado"))


df_canal = get_most_frequent("canal")
df_dispositivo = get_most_frequent("dispositivo")

fato_clickstream_navegacao = df_agregado.join(
    df_canal, "id_cliente", "left").join(df_dispositivo, "id_cliente", "left")
fato_clickstream_navegacao = fato_clickstream_navegacao.withColumn(
    "timestamp_ingestion_gold", F.current_timestamp()
)

fato_clickstream_navegacao.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{gold_schema_name}.fato_clickstream_navegacao")

print(f"✅ Tabela {catalog}.{gold_schema_name}.fato_clickstream_navegacao criada com sucesso!\n")

✅ Tabela v_commerce.gold.fato_clickstream_navegacao criada com sucesso!



In [0]:
# gold.fato_suporte_ticket

df_join_tickets_pedidos = tb_suporte_tickets_silver.join(
    tb_pedidos_silver.select("id_pedido", "id_produto"),
    "id_pedido",
    "left"
)

fato_suporte_ticket = df_join_tickets_pedidos.withColumn(
    "status", F.when(F.col("data_resolucao").isNull(),
                     "aberto").otherwise("resolvido")
).select(
    F.col("id_ticket"),
    F.col("id_cliente"),
    F.col("id_pedido"),
    F.col("id_produto"),
    F.col("data_abertura"),
    F.col("data_resolucao"),
    F.col("tempo_resolucao_horas"),
    F.col("status"),
    F.col("tipo_problema"),
    F.col("agente_suporte"),
    F.col("nota_avaliacao"),
    F.current_timestamp().alias("timestamp_ingestion_gold")
)

fato_suporte_ticket.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{gold_schema_name}.fato_suporte_ticket")

print(f"✅ Tabela {catalog}.{gold_schema_name}.fato_suporte_ticket criada com sucesso!\n")

✅ Tabela v_commerce.gold.fato_suporte_ticket criada com sucesso!



In [0]:
# gold.gold_satisfacao_problema

gold_satisfacao_problema = tb_suporte_tickets_silver.groupBy("tipo_problema").agg(
    F.count("id_ticket").alias("volume_tickets"),
    F.round(F.avg("nota_avaliacao"), 2).alias("nota_media_satisfacao"),
    F.round(F.avg("tempo_resolucao_horas"), 2).alias(
        "tempo_medio_resolucao_horas")
)

gold_satisfacao_problema.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{gold_schema_name}.gold_satisfacao_problema")

print(f"✅ Tabela {catalog}.{gold_schema_name}.gold_satisfacao_problema criada com sucesso")

✅ Tabela v_commerce.gold.gold_satisfacao_problema criada com sucesso


In [0]:
# gold.fato_vendas

fato_vendas = (
    pedidos
    .join(
        tb_catalogo_produtos_silver.select("id_produto", "preco"),
        "id_produto",
        "left"
    )

    .select(
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),
        F.col("data_pedido").alias("id_data"),
        F.col("quantidade").alias("quantidade_vendas"),
        F.col("preco").alias("valor_unitario"),
        # valor real do pedido vindo da silver
        F.col("valor_pedido").alias("valor_total_venda"),
        F.col("status"),
        F.col("metodo_pagamento"),

        # TIMESTAMP DE PROCESSAMENTO
        F.current_timestamp().alias("timestamp_ingestion_gold")
    )
)

fato_vendas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.fato_vendas")

print(f"✅ Tabela {catalog}.{gold_schema_name}.fato_vendas criada com sucesso")

✅ Tabela v_commerce.gold.fato_vendas criada com sucesso


In [0]:
# gold.dim_cliente

pedidos_cliente = (
    fato_vendas
    .groupBy("id_cliente")
    .agg(
        F.countDistinct("id_pedido").alias("frequencia"),  # Quantidade de pedidos
        F.round(F.sum("valor_total_venda"), 2).alias("valor_monetario"),  # Valor total gasto
        F.max("id_data").alias("data_ultima_compra")  # Data da última compra
    )
    .withColumn("recencia_dias", F.datediff(F.current_date(), F.col("data_ultima_compra")))  # Dias desde última compra
)


# CALCULAR SCORES RFM
window_base = Window.partitionBy(F.lit(1))

rfm_scores = (
    pedidos_cliente
    .withColumn(
        "r_score",
        6 - F.ntile(5).over(window_base.orderBy(F.col("recencia_dias").asc()))  # Score de Recência
    )
    .withColumn(
        "f_score",
        F.ntile(5).over(window_base.orderBy(F.col("frequencia").asc()))  # Score de Frequência
    )
    .withColumn(
        "m_score",
        F.ntile(5).over(window_base.orderBy(F.col("valor_monetario").asc()))  # Score de Monetário
    )
)

# AGREGAR TICKETS DE SUPORTE POR CLIENTE


tickets_cliente = (
    spark.table("v_commerce.silver.tb_suporte_tickets")
    .groupBy("id_cliente")
    .agg(F.count("id_ticket").alias("qtd_tickets_suporte"))  # Quantidade de tickets
)

# AGREGAR AVALIAÇÕES POR CLIENTE
avaliacoes_cliente = (
    spark.table("v_commerce.silver.tb_avaliacoes")
    .groupBy("id_cliente")
    .agg(F.round(F.avg("nota_produto"), 1).alias("media_estrelas_dadas"))  # Média de estrelas dadas
)

# MAPEAR REGIÃO POR ESTADO
mapeamento_regiao = (
    F.when(F.col("estado").isin("AC","AP","AM","PA","RO","RR","TO"), "Norte")
     .when(F.col("estado").isin("AL","BA","CE","MA","PB","PE","PI","RN","SE"), "Nordeste")
     .when(F.col("estado").isin("DF","GO","MT","MS"), "Centro-Oeste")
     .when(F.col("estado").isin("ES","MG","RJ","SP"), "Sudeste")
     .when(F.col("estado").isin("PR","RS","SC"), "Sul")
     .otherwise("Não informado")
)

# CRIAR DIMENSÃO CLIENTE GOLD
dim_cliente = (
    tb_clientes_silver
    .join(rfm_scores, "id_cliente", "left")  # Adiciona scores RFM
    .join(tickets_cliente, "id_cliente", "left")  # Adiciona tickets de suporte
    .join(avaliacoes_cliente, "id_cliente", "left")  # Adiciona avaliações

    .select(
        F.col("id_cliente"),
        F.concat_ws(" ", F.col("nome"), F.col("sobrenome")).alias("nome_cliente"),
        F.col("cidade"),
        F.col("estado"),
        mapeamento_regiao.alias("regiao"),  # Região do cliente

        F.coalesce(F.col("frequencia"), F.lit(0)).alias("qtd_pedidos_realizados"),
        F.coalesce(F.col("valor_monetario"), F.lit(0.0)).alias("total_gasto_brl"),
        F.coalesce(F.col("qtd_tickets_suporte"), F.lit(0)).alias("qtd_tickets_suporte"),

        F.col("data_ultima_compra"),

        F.coalesce(F.col("media_estrelas_dadas"), F.lit(0.0)).alias("media_estrelas_dadas"),

        # Segmentação RFM
        F.when(
            F.col("r_score").isNull(),
            "Inativo"
        )
        .when(
            (F.col("r_score") >= 4) &
            (F.col("f_score") >= 4) &
            (F.col("m_score") >= 4),
            "Campeão"
        )
        .when(
            (F.col("f_score") >= 4) &
            (F.col("m_score") >= 4),
            "Cliente VIP"
        )
        .when(
            (F.col("r_score") <= 2) &
            (F.col("f_score") >= 4),
            "Em risco"
        )
        .when(
            (F.col("f_score") >= 4),
            "Cliente fiel"
        )
        .when(
            (F.col("f_score") >= 3),
            "Cliente regular"
        )
        .when(
            (F.col("r_score") == 1) &
            (F.col("f_score") <= 2) &
            (F.col("m_score") <= 2),
            "Inativo"
        )
        .when(
            (F.col("r_score") >= 4) &
            (F.col("f_score") <= 2),
            "Novo cliente"
        )
        .otherwise("Cliente regular")
        .alias("segmento_rfm"),

        F.current_timestamp().alias("timestamp_ingestion_gold")  # Timestamp de ingestão
    )
)

# SALVAR DIMENSÃO CLIENTE GOLD
dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.dim_cliente")

print(f"✅ Tabela {catalog}.{gold_schema_name}.dim_cliente criada com sucesso")

✅ Tabela v_commerce.gold.dim_cliente criada com sucesso


In [0]:
# gold.gold_satisfacao_agente

gold_satisfacao_agente = (
    tickets
    .filter(F.col("agente_suporte").isNotNull())
    .groupBy("agente_suporte")
    .agg(
        F.count("*").alias("qtd_tickets_resolvidos"),
        F.round(
            F.avg(
                F.when(F.col("nota_avaliacao") != -1, F.col("nota_avaliacao"))
            ), 2
        ).alias("nota_media_satisfacao"),
        F.round(F.avg("tempo_resolucao_horas"), 2).alias("tempo_medio_resolucao"),
        
   
        # TIMESTAMP DE PROCESSAMENTO
        F.current_timestamp().alias("timestamp_ingestion_gold")
    )
)

gold_satisfacao_agente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.gold_satisfacao_agente")
print(f"✅ Tabela {catalog}.{gold_schema_name}.gold_satisfacao_agente criada com sucesso")

✅ Tabela v_commerce.gold.gold_satisfacao_agente criada com sucesso


In [ ]:
def normalize_text(column):
    return F.lower(
        F.translate(
            F.trim(column),
            "áàãâäéèêëíìîïóòõôöúùûüçÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ",
            "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
        )
    )

In [0]:
# gold.dim_categoria
# Origem: silver.tb_catalogo_produtos

catalogo = spark.table("v_commerce.silver.tb_catalogo_produtos")

imagem_categoria = (
    F.when(F.col("nome_categoria") == "Eletrônicos", "https://static.vecteezy.com/ti/vetor-gratis/p3/4686347-electronic-devices-set-of-vector-icons-vetor.jpg")
     .when(F.col("nome_categoria") == "Vestuário", "https://static.vecteezy.com/system/resources/previews/019/516/083/original/clothing-store-icons-set-black-on-a-white-background-vector.jpg")
     .when(F.col("nome_categoria") == "Casa", "https://static.vecteezy.com/system/resources/previews/008/891/612/non_2x/towel-icons-set-outline-style-vector.jpg")
     .when(F.col("nome_categoria") == "Esportes", "https://static.vecteezy.com/system/resources/previews/005/766/036/original/set-of-modern-graphics-of-a-collection-of-sport-icons-vector.jpg")
     .when(F.col("nome_categoria") == "Móveis", "https://static.vecteezy.com/ti/vetor-gratis/p1/6820077-conjunto-de-icones-simples-em-um-tema-moveis-casa-design-interior-sinal-simbolo-objeto-ilustracao-icones-pretos-isolados-contra-fundo-branco-vetor.jpg")
     .when(F.col("nome_categoria") == "Brinquedos", "https://static.vecteezy.com/system/resources/previews/025/460/753/non_2x/pack-of-childhood-toys-flat-icons-vector.jpg")
     .when(F.col("nome_categoria") == "Automotivo", "https://static.vecteezy.com/system/resources/previews/000/459/718/non_2x/vector-car-repair-icons-black.jpg")
     .when(F.col("nome_categoria") == "Beleza", "https://www.kindpng.com/picc/m/59-592719_icons-of-post-office-hd-png-download.png")
     .otherwise("https://static.vecteezy.com/system/resources/previews/041/731/191/non_2x/more-icon-vector.jpg")
)
window_categoria = Window.orderBy("nome_categoria")

gold_categoria = (
    catalogo
    .filter(F.col("categoria").isNotNull())
    .withColumnRenamed("categoria", "nome_categoria")
    .groupBy("nome_categoria")
    .agg(
        F.count("id_produto")                                              .alias("total_produtos"),
        F.sum(F.when(F.col("ativo") == "Sim", 1).otherwise(0))            .alias("total_produtos_ativos"),
        F.sum(F.when(F.col("tem_estoque") == "Sim", 1).otherwise(0))      .alias("total_com_estoque"),
        F.round(F.avg("preco"), 2)                                         .alias("preco_medio"),
        F.min(F.when(F.col("preco") != -1, F.col("preco")))               .alias("preco_minimo"),
        F.max("preco")                                                     .alias("preco_maximo"),
        F.round(F.avg("peso_kg"), 2)                                       .alias("peso_medio_kg"),
        F.sum(F.when(F.col("precisa_revisao") == "Sim", 1).otherwise(0))  .alias("total_precisa_revisao"),
    )
    .withColumn(
        "id_categoria",
        F.concat(
            F.lit("CATEG-"),
            F.lpad(
                F.row_number().over(window_categoria).cast("string"), 4,"0")
        )
    )
    .withColumn(
    "slug_categoria",
    F.regexp_replace(
        normalize_text(F.col("nome_categoria")), " ", "-")
    )
    .select(

        # PK
        F.col("id_categoria"),
        
        F.col("nome_categoria"),
        F.col("slug_categoria"),

        imagem_categoria.alias("imagem_url"),

        # métricas
        F.col("total_produtos"),
        F.col("total_produtos_ativos"),
        F.col("total_com_estoque"),
        F.col("preco_medio"),
        F.col("preco_minimo"),
        F.col("preco_maximo"),
        F.col("peso_medio_kg"),
        F.col("total_precisa_revisao"),

        F.current_timestamp().alias("timestamp_ingestion_gold"),
    )
)

gold_categoria.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.gold_categoria")
print(f"✅ Tabela {catalog}.{gold_schema_name}.gold_categoria criada com sucesso!")

✅ Tabela v_commerce.gold.dim_categoria criada com sucesso!


In [0]:
spark.sql("""
    OPTIMIZE v_commerce.gold.fato_vendas
    ZORDER BY (id_cliente, id_produto)
""")

spark.sql("""
    OPTIMIZE v_commerce.gold.fato_suporte_ticket
    ZORDER BY (id_cliente, id_pedido)
""")

spark.sql("""
    OPTIMIZE v_commerce.gold.fato_avaliacoes_pedido 
    ZORDER BY (id_produto, id_cliente)
""")

spark.sql("""
    OPTIMIZE v_commerce.gold.fato_clickstream_navegacao
    ZORDER BY (id_cliente)
""")
     

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,